# Gradient method demo

This notebook walks through first-order optimizers in
`trigger_agent_misc.optimization.gradient_method`. Run from the repo root after `uv sync`.

We minimize a finite-sum quadratic

$$f(x) = \frac{1}{m}\sum_{i=1}^{m} \tfrac{1}{2}(x - c_i)^2$$

Full-batch methods use $\nabla f(x)$. SGD and mini-batch SGD use sampled $\nabla f_i(x)$.


In [ ]:
from trigger_agent_misc.optimization.gradient_method import (
    gradient_descent,
    heavy_ball_method,
    minibatch_sgd,
    momentum,
    nesterov_accelerated_gradient,
    stochastic_gradient_descent,
)

CENTERS = (-2.0, 0.5, 1.5)


def objective(x):
    return sum(0.5 * (x[0] - c) ** 2 for c in CENTERS) / len(CENTERS)


def grad(x):
    mean = sum(x[0] - c for c in CENTERS) / len(CENTERS)
    return (mean,)


def grad_at_index(i, x):
    return (x[0] - CENTERS[i],)


In [ ]:
import random

x0 = (3.0,)
runs = {
    "gradient_descent": gradient_descent(x0, objective, grad, learning_rate=0.2),
    "sgd": stochastic_gradient_descent(
        x0, objective, grad_at_index, len(CENTERS), learning_rate=0.05, rng=random.Random(0)
    ),
    "minibatch_sgd": minibatch_sgd(
        x0,
        objective,
        grad_at_index,
        len(CENTERS),
        batch_size=2,
        learning_rate=0.1,
        rng=random.Random(1),
    ),
    "momentum": momentum(x0, objective, grad, learning_rate=0.05, beta=0.9),
    "nesterov": nesterov_accelerated_gradient(
        x0, objective, grad, learning_rate=0.05, beta=0.9
    ),
    "heavy_ball": heavy_ball_method(
        x0, objective, grad, learning_rate=0.05, beta=0.85
    ),
}
for name, result in runs.items():
    print(name, result.x[0], result.value, result.iterations)


See also ``evals/gradient_method_eval.py`` for scripted metrics used in experiments.
